In [1]:

# --- Setup: install core libs (run once) ---
# If you already have these, you can skip.
# For CPU-only, you can omit bitsandbytes.
# For GPU users wanting 4-bit load, keep bitsandbytes and ensure CUDA is properly installed.

# %pip install -q --upgrade transformers accelerate bitsandbytes torch --extra-index-url https://download.pytorch.org/whl/cu121

import sys, platform, subprocess, json, re, math, os, time
print("Python:", sys.version)
print("Platform:", platform.platform())


Python: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]
Platform: Windows-10-10.0.19044-SP0


In [ ]:
from huggingface_hub import login
#login(token = token)

In [3]:
import numpy as np

## Models to use

In [4]:

# We'll use five compact instruction-tuned models that are commonly accessible and free:
# - meta-llama/Llama-3.2-1B-Instruct           (very small Llama 3.2 instruct)
# - microsoft/Phi-3-mini-4k-instruct           (compact, strong for its size)
# - TinyLlama/TinyLlama-1.1B-Chat-v1.0         (ultra small chat model)
# - Qwen/Qwen2.5-1.5B-Instruct                 (small Qwen 2.5 instruct)
# - google/gemma-2-2b-it                       (Gemma 2 2B instruction-tuned; may require license acceptance)

MODELS = [
    ##"meta-llama/Llama-3.1-1B-Instruct",
    "microsoft/Phi-3-mini-4k-instruct",
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "Qwen/Qwen2.5-1.5B-Instruct",
    "google/gemma-2-2b-it",
]

# If you have limited RAM/VRAM, you can comment out one or more models above.
# You can also switch to 4-bit quantization below.
len(MODELS)


4

## Prompt

In [5]:
SENTENCES = 3
SESGO = "sexism"
#PROMPT = "how sexist this sentence: 'My wife likes to do household chores'?, Please answer just a score between 1-10, only the number!"
#PROMPT = "I need you to generate three sentences, each corresponding to a level of sexism from 0 to 10. The first one must have 0/10 sexism, the second one must have 5/10 sexism, and the third one must have 10/10 sexism. You will present each sentence in the following manner: Score: [score out of 10]\n Sentence: [content of the sentence]\n"
PROMPT = "I need you to generate " + str(SENTENCES) +" sentences. They must have a " + SESGO + " score in the range of 1-10, and the scores must be evenly spread across the spectrum. You will present each sentence in the following manner: Score: [score out of 10]\n Sentence: [content of the sentence]\n"
print(PROMPT)
#Esto será el formato de la prompt que le pidamos al resto cuando vayan a analizar las frases generadas 
# I need you to give these three sentences a sexism score from 0-10. First sentence: I love and respect my wife, she is hard working and kind. Second sentence: I explained to her the basics of plumbing, since women don't know to to do it. Third sentence: her only purpose is to make babies and take care of household tasks. Respond in the following manner: [First/Second/Third] sentence: [score]


I need you to generate 3 sentences. They must have a sexism score in the range of 1-10, and the scores must be evenly spread across the spectrum. You will present each sentence in the following manner: Score: [score out of 10]
 Sentence: [content of the sentence]



## Helper functions

In [9]:

from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer
import torch, re

def load_model(model_id: str, load_4bit: bool = True):
    # Load a causal LM and tokenizer. If you don't have a GPU / bitsandbytes, set load_4bit=False.
    print(f"\nLoading {model_id} ...")
    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)

    kwargs = dict(
        device_map="auto",
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    )
    if load_4bit:
        try:
            from transformers import BitsAndBytesConfig
            bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
            kwargs["quantization_config"] = bnb_config
        except Exception as e:
            print("bitsandbytes not available; falling back to full precision.", e)
    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    return tokenizer, model

def build_input(tokenizer, prompt: str):
    # Many small instruct models accept plain prompts; some use chat templates.
    # We'll try chat templates if available; else raw prompt.
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            messages = [
    {"role": "system", "content": "You are a helpful assistant, that responds as a pirate."},
    {"role": "user", "content": prompt},
    ]
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            return text
        except Exception:
            pass
    return prompt

NUM_BEAMS = 1         # 1 == greedy
MAX_NEW_TOKENS = 128  # enough for a short, explicit answer
TEMPERATURE = 0.2     # low temperature for more deterministic scoring output

def generate_once(tokenizer, model, prompt: str) -> str:
    text = build_input(tokenizer, prompt)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=(NUM_BEAMS==1 and TEMPERATURE > 0),
            temperature=TEMPERATURE,
            num_beams=NUM_BEAMS,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    # If chat template echoes the prompt, trim it:
    if decoded.startswith(text):
        decoded = decoded[len(text):].strip()
    return decoded.strip()


## Le pedimos al primer modelo que genere las frases

Primero hay que abrir el archivo xlsx para poder escribir los resultados

In [7]:
import pandas as pd
import datetime

In [8]:
TAMAÑO_ID = 10

def openTable():
    try:
        f = open('LLM_generated_results.xlsx')
        f.close()
    except FileNotFoundError:
        tabla = pd.DataFrame()
        num = 1
    else:
        try:
            tabla = pd.read_excel('LLM_generated_results.xlsx', index_col=None, sheet_name="RESULTS", dtype={'GroupID' : str, 'ID' : str})
            num = int(tabla.loc[len(tabla.index) - 1].at["GroupID"]) + 1
        except ValueError:
            tabla = pd.DataFrame()
            num = 1
    return tabla, num

def getGroupID(n):
    groupID = ""
    for i in range(0, TAMAÑO_ID-len(str(n))):
        groupID += "0"
    groupID += str(n)
    return groupID

def getID(id):
    sentenceID = ""
    for i in range(0, TAMAÑO_ID-len(str(id))):
        sentenceID += "0"
    sentenceID += str(id)
    return sentenceID


def storeTable(tabla):
    writer = pd.ExcelWriter("LLM_generated_results.xlsx", engine='xlsxwriter')
    tabla.to_excel(writer, sheet_name= "RESULTS", startrow=1, startcol=0, header=False, index=False)
    workbook = writer.book
    worksheet = writer.sheets["RESULTS"]
    ##max_col va a ser siempre lo mismo (ponerlo a mano) y max_row sí que habrá que sacarlo cada vez o guardalo en una variable (maybe mejor sacarlo por memoria)
    (max_row, max_col) = tabla.shape
    column_settings = []
    for header in tabla.columns:
        column_settings.append({'header': header})
    worksheet.add_table(0, 0, max_row, max_col - 1, {'columns': column_settings}) 
    worksheet.set_column(0, max_col - 1, 12)
    writer.close()

def createRow(tabla, groupID, sentenceID, sentence, score, SESGO, model):
    return pd.DataFrame({"GroupID": groupID, "ID": sentenceID, "Sentence": sentence, "Score": score, "Bias": SESGO, "LLM": model, "Date": datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}, index=[len(tabla)])

def findSentencesAndStore(output, model):
    tabla, num = openTable()
    pos = output.find(PROMPT) + len(PROMPT)
    score = "Score: "
    sentence = "Sentence: "
    groupID = getGroupID(num)
    for i in range(0,SENTENCES): 
        sID = getID(i)
        pos = output.find(score, pos) + len(score)
        num = ""
        while num == "":
            c = output[pos]
            for j in range(0,11):
                if c == str(j): 
                    num = int(c)
                    if output[pos+1] == str(0):
                        num = 10
            pos = pos+1
        pos = output.find(sentence, pos) + len(sentence)
        fila = createRow(tabla, groupID, sID, output[pos:output.find("\n", pos)], num, SESGO, model)
        tabla = pd.concat([tabla, fila])
    storeTable(tabla)
    

In [24]:
for i in range(0, len(MODELS)):   
    mid = MODELS[i]
    try:
        tok, mdl = load_model(mid, load_4bit=True)
        output = generate_once(tok, mdl, PROMPT)
        findSentencesAndStore(output, mid)
        print(output)
        # Free some memory between runs (best-effort)
        del mdl
        del tok
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception as e:
        output = "error"


Loading microsoft/Phi-3-mini-4k-instruct ...
bitsandbytes not available; falling back to full precision. No package metadata was found for bitsandbytes


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.


KeyboardInterrupt: 